# Whisper Medium Training - Sundanese ASR

Fine-tuning Whisper Medium on OpenSLR36 Sundanese dataset.

**Requirements:**
- GPU P100 (16GB VRAM)
- Dataset: `openslr36-sundanese-asr-prepared`

## Cell 1: Install Dependencies

Run this cell, then **Restart Session** before continuing.

In [1]:
!pip uninstall -y datasets
!pip install datasets==2.21.0 soundfile librosa transformers evaluate jiwer
print("\n" + "="*50)
print("DONE! Now click: Runtime -> Restart Session")
print("Then run Cell 2")
print("="*50)

Found existing installation: datasets 2.21.0
Uninstalling datasets-2.21.0:
  Successfully uninstalled datasets-2.21.0
  Using cached datasets-2.21.0-py3-none-any.whl.metadata (21 kB)
Using cached datasets-2.21.0-py3-none-any.whl (527 kB)

DONE! Now click: Runtime -> Restart Session
Then run Cell 2


## Cell 2: Configuration

In [2]:
import os
import sys
import json
import torch
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")

@dataclass
class Config:
    model_name: str = "openai/whisper-small"  # CHANGED
    language: str = "sun"
    task: str = "transcribe"
    output_dir: Path = KAGGLE_WORKING / "whisper-small-sundanese"
    
    max_steps: int = 5000
    batch_size: int = 4
    eval_batch_size: int = 4
    gradient_accumulation: int = 4  # Effective batch = 16
    learning_rate: float = 1e-5
    warmup_steps: int = 500
    
    save_steps: int = 500
    eval_steps: int = 500
    logging_steps: int = 50
    
    fp16: bool = True
    gradient_checkpointing: bool = True

config = Config()
print(f"Model: {config.model_name}")
print(f"Batch: {config.batch_size} x {config.gradient_accumulation} = {config.batch_size * config.gradient_accumulation}")

Model: openai/whisper-small
Batch: 4 x 4 = 16


## Cell 3: Helper Functions

In [3]:
import os

def find_dataset():
    """Find dataset in Kaggle input."""
    search_paths = [
        KAGGLE_INPUT / "openslr36-sundanese-asr-prepared" / "kaggle_dataset" / "manifests",
        KAGGLE_INPUT / "openslr36-sundanese-asr-prepared" / "manifests",
        KAGGLE_INPUT / "openslr36-sundanese-asr-prepared",
    ]
    
    for path in search_paths:
        if (path / "train.json").exists():
            print(f"Found dataset: {path}")
            return path
    
    for d in KAGGLE_INPUT.iterdir():
        if d.is_dir():
            for sub in ["kaggle_dataset/manifests", "manifests", ""]:
                check = d / sub if sub else d
                if (check / "train.json").exists():
                    print(f"Found dataset: {check}")
                    return check
    
    raise FileNotFoundError("Dataset not found!")


def fix_audio_path(old_path, audio_base):
    """Fix audio path to match actual Kaggle structure."""
    if "openslr36-audio/" in old_path:
        rel = old_path.split("openslr36-audio/")[-1]
    elif "openslr36-audio\\" in old_path:
        rel = old_path.split("openslr36-audio\\")[-1]
    else:
        rel = old_path.split("/audio/")[-1] if "/audio/" in old_path else old_path
    rel = rel.replace("\\", "/")
    return f"{audio_base}/{rel}"


def load_manifest(path: Path, audio_base: Path = None, validate_audio: bool = True):
    """Load dataset from manifest JSON."""
    from datasets import Dataset, Audio
    
    print(f"Loading {path.name}...")
    
    with open(path, 'r', encoding='utf-8') as f:
        entries = json.load(f)
    
    audio_base_str = str(audio_base) if audio_base else None
    valid_entries = []
    missing_count = 0
    
    for e in entries:
        ap = e['audio_path']
        if audio_base_str:
            fixed_path = fix_audio_path(ap, audio_base_str)
        else:
            fixed_path = ap.replace('\\', '/')
        
        if validate_audio and not os.path.exists(fixed_path):
            missing_count += 1
            continue
        
        valid_entries.append({'audio': fixed_path, 'transcription': e['transcription']})
    
    if missing_count > 0:
        print(f"  ⚠ Skipped {missing_count:,} missing files")
    
    dataset = Dataset.from_dict({
        'audio': [e['audio'] for e in valid_entries],
        'transcription': [e['transcription'] for e in valid_entries],
    })
    dataset = dataset.cast_column('audio', Audio(sampling_rate=16000))
    print(f"  ✓ Loaded {len(dataset):,} samples")
    return dataset


def prepare_batch(batch, processor):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["labels"] = processor.tokenizer(batch["transcription"]).input_ids
    return batch


@dataclass
class DataCollator:
    processor: Any
    
    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]
        
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        
        batch["labels"] = labels
        return batch


def compute_wer(pred, processor, metric):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {"wer": 100 * metric.compute(predictions=pred_str, references=label_str)}

print("Helper functions loaded!")

Helper functions loaded!


## Cell 4: Check GPU & Load Dataset

In [4]:
# Check GPU
print("="*60)
print("GPU Check")
print("="*60)

if not torch.cuda.is_available():
    print("ERROR: No GPU detected!")
    print("Go to: Settings -> Accelerator -> GPU P100")
else:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Find dataset
print("\n" + "="*60)
print("Dataset")
print("="*60)

dataset_dir = find_dataset()
audio_base = dataset_dir.parent / "audio" if dataset_dir.name == "manifests" else dataset_dir / "audio"

if audio_base.exists():
    print(f"Audio base: {audio_base}")
    print("\nAvailable audio folders:")
    folders = sorted([f for f in audio_base.iterdir() if f.is_dir()])
    for folder in folders:
        file_count = sum(1 for _ in folder.rglob("*.flac"))
        print(f"  {folder.name}: {file_count:,} files")
    print(f"\nTotal folders: {len(folders)}")
else:
    print(f"Warning: Audio not found at {audio_base}")
    audio_base = None

GPU Check
GPU: Tesla P100-PCIE-16GB
VRAM: 17.1 GB

Dataset
Found dataset: /kaggle/input/openslr36-sundanese-asr-prepared/kaggle_dataset/manifests
Audio base: /kaggle/input/openslr36-sundanese-asr-prepared/kaggle_dataset/audio

Available audio folders:
  asr_sundanese_0: 2,481 files
  asr_sundanese_1: 2,681 files
  asr_sundanese_2: 2,897 files
  asr_sundanese_3: 3,024 files
  asr_sundanese_4: 3,322 files
  asr_sundanese_5: 3,733 files
  asr_sundanese_6: 4,160 files
  asr_sundanese_8: 4,421 files
  asr_sundanese_9: 4,691 files
  asr_sundanese_a: 5,313 files
  asr_sundanese_b: 6,000 files
  asr_sundanese_c: 6,795 files
  asr_sundanese_d: 7,750 files
  asr_sundanese_e: 8,880 files
  asr_sundanese_f: 10,122 files

Total folders: 15


## Cell 5: Load Model & Processor

In [5]:
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
import evaluate
import os

# Disable tokenizer parallelism warning
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print(f"Loading {config.model_name}...")

processor = WhisperProcessor.from_pretrained(
    config.model_name,
    language=config.language,
    task=config.task
)

model = WhisperForConditionalGeneration.from_pretrained(config.model_name)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.config.use_cache = False  # Required for gradient checkpointing

# Enable gradient checkpointing properly
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

print(f"Model loaded! Parameters: {model.num_parameters():,}")

2026-01-01 16:30:15.702886: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767285015.725104     276 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767285015.731839     276 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767285015.751275     276 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767285015.751302     276 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767285015.751304     276 computation_placer.cc:177] computation placer alr

Loading openai/whisper-small...


preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

Model loaded! Parameters: 241,734,912


## Cell 6: Load & Prepare Datasets

In [6]:
print("Loading datasets...")
train_data = load_manifest(dataset_dir / "train.json", audio_base, validate_audio=True)
eval_data = load_manifest(dataset_dir / "validation.json", audio_base, validate_audio=True)

# Limit dataset to prevent OOM
MAX_TRAIN = 15000  # Reduce if still crashes
MAX_EVAL = 2000

if len(train_data) > MAX_TRAIN:
    print(f"\n⚠ Limiting train from {len(train_data):,} to {MAX_TRAIN:,} samples")
    train_data = train_data.shuffle(seed=42).select(range(MAX_TRAIN))

if len(eval_data) > MAX_EVAL:
    print(f"⚠ Limiting eval from {len(eval_data):,} to {MAX_EVAL:,} samples")
    eval_data = eval_data.shuffle(seed=42).select(range(MAX_EVAL))

print("\nPreparing datasets (batched to save memory)...")

train_data = train_data.map(
    lambda x: prepare_batch(x, processor),
    remove_columns=train_data.column_names,
    writer_batch_size=500,
    desc="Preparing train",
)
eval_data = eval_data.map(
    lambda x: prepare_batch(x, processor),
    remove_columns=eval_data.column_names,
    writer_batch_size=500,
    desc="Preparing eval",
)

print(f"\n✓ Train: {len(train_data):,} samples")
print(f"✓ Eval: {len(eval_data):,} samples")

Loading datasets...
Loading train.json...
  ⚠ Skipped 103,411 missing files
  ✓ Loaded 60,991 samples
Loading validation.json...
  ⚠ Skipped 12,931 missing files
  ✓ Loaded 7,619 samples

⚠ Limiting train from 60,991 to 15,000 samples
⚠ Limiting eval from 7,619 to 2,000 samples

Preparing datasets (batched to save memory)...


Preparing train:   0%|          | 0/15000 [00:00<?, ? examples/s]

Preparing eval:   0%|          | 0/2000 [00:00<?, ? examples/s]


✓ Train: 15,000 samples
✓ Eval: 2,000 samples


## Cell 7: Setup Trainer

In [7]:
config.output_dir.mkdir(parents=True, exist_ok=True)

data_collator = DataCollator(processor=processor)
metric = evaluate.load("wer")

training_args = Seq2SeqTrainingArguments(
    output_dir=str(config.output_dir),
    max_steps=config.max_steps,
    per_device_train_batch_size=config.batch_size,
    per_device_eval_batch_size=config.eval_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation,
    learning_rate=config.learning_rate,
    warmup_steps=config.warmup_steps,
    fp16=config.fp16,
    eval_strategy="steps",
    eval_steps=config.eval_steps,
    save_strategy="steps",
    save_steps=config.save_steps,
    logging_steps=config.logging_steps,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=225,
    dataloader_num_workers=2,
    report_to=["tensorboard"],
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=eval_data,
    data_collator=data_collator,
    compute_metrics=lambda pred: compute_wer(pred, processor, metric),
    processing_class=processor.feature_extractor,
)

print("Trainer ready!")
print(f"Output: {config.output_dir}")

Trainer ready!
Output: /kaggle/working/whisper-small-sundanese


## Cell 8: Train!

In [ ]:
print("="*60)
print("Starting Training")
print("="*60)
print(f"Model: {config.model_name}")
print(f"Batch: {config.batch_size} x {config.gradient_accumulation} = {config.batch_size * config.gradient_accumulation}")
print(f"Steps: {config.max_steps}")
print(f"LR: {config.learning_rate}")
print("="*60 + "\n")

trainer.train()

Starting Training
Model: openai/whisper-small
Batch: 4 x 4 = 16
Steps: 5000
LR: 1e-05



You're using a WhisperTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a WhisperTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss,Validation Loss,Wer
500,0.120600,0.112306,12.189161
1000,0.031500,0.060033,8.700434
1500,0.029700,0.050301,9.568033
2000,0.005300,0.044921,6.781939
2500,0.004600,0.044203,10.124030
3000,0.001700,0.042385,8.401051
3500,0.001800,0.042468,6.885807
4000,0.000700,0.042265,7.686198
4500,0.000700,0.042417,7.606770


You're using a WhisperTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a WhisperTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
The attention mask is not set and 

## Cell 9: Save Model

In [ ]:
print("Saving model...")
trainer.save_model()
processor.save_pretrained(config.output_dir)

print(f"\nModel saved to: {config.output_dir}")
print("\nTo download:")
print("1. Click 'Save Version' (top right)")
print("2. Select 'Save & Run All'")
print("3. After completion, go to Output tab")

## Cell 10: Evaluate on Test Set (Optional)

In [ ]:
test_file = dataset_dir / "test.json"

if test_file.exists():
    print("Evaluating on test set...")
    test_data = load_manifest(test_file, audio_base)
    test_data = test_data.map(
        lambda x: prepare_batch(x, processor),
        remove_columns=test_data.column_names,
        num_proc=2,
    )
    
    results = trainer.evaluate(test_data)
    print(f"\nTest WER: {results['eval_wer']:.2f}%")
    
    with open(config.output_dir / "test_results.json", 'w') as f:
        json.dump(results, f, indent=2)
else:
    print("No test.json found, skipping test evaluation.")